# Sandbox — Silver Sellers

Exploração por trás de `silver_rules.tratar_sellers`. Padrão análogo ao de `customers`: **normalização de texto geográfico** — cidade em minúsculo, UF em maiúsculo.

> **Atenção:** `seller_zip_code_prefix` é mantido como **string**, não int — o CEP brasileiro pode ter zeros à esquerda (ex: `01234`), que um cast para inteiro destruiria.

In [ ]:
from utils.spark_utils import create_spark_session
from pyspark.sql.functions import col, trim, lower, upper, current_timestamp

spark = create_spark_session('Sandbox-Silver-Sellers')

df_bronze = spark.read.parquet('s3a://bronze/olist/sellers')
df_bronze.show(5, truncate=False)
df_bronze.printSchema()

Conferindo se há inconsistência de caixa em `seller_state` (ex: 'sp' e 'SP' conviverem), o que justifica o `upper`.

In [ ]:
df_bronze.select('seller_state').distinct().show()

In [ ]:
df_silver = (df_bronze
    .withColumn('seller_id', trim(col('seller_id')))
    .withColumn('seller_zip_code_prefix', trim(col('seller_zip_code_prefix')))
    .withColumn('seller_city', lower(trim(col('seller_city'))))
    .withColumn('seller_state', upper(trim(col('seller_state'))))
    .withColumn('dt_criacao_silver', current_timestamp()))

df_silver.printSchema()
df_silver.show(5, truncate=False)

**Lógica promovida para `silver_rules.tratar_sellers`.**

In [ ]:
spark.stop()